# 리포트 18 — 가림 판정은 Sionna 광선엔진이 하고, 면적분은 우리 커널이 한다

> ### 한 일
> **상용 고주파 솔버의 순서 그대로 광선으로 조명면을 찾고 그 면 위에서 부품별 재질 PO 를 적분해 σ 를 냈다.**

### 결과
1. 첫 충돌 탐색과 가림은 Sionna 가 이미 들고 있는 Mitsuba/OptiX 엔진이 하고, 표면전류 적분과 σ 출력은 우리가 얹는다 — 그 문서에 `physical optics` 는 0 회 [^1] 나온다.
2. 조명원을 방위 280° [^2] · 고각 15° [^3] 에 두면 조명원을 향한 외피의 29 [^4]~47% [^5] 가 기체 자신에 가려 있다.
3. 그 가림을 끄면 방위평균 σ 가 최대 6.63 dB [^6] (Matrice 4E ⭐ [^7], 닫힌 동체)까지 부풀고, 열린 프레임인 S1000+ [^8] 에서는 0.11 dB [^9] 다. 기체 7 종 [^10] 전부에서 이 값이 이산화 바닥(최대 0.071 dB [^11]) 위에 있다 — 가림은 수치잡음이 아니라 물리다.
4. 금속 4그룹만 남긴 메쉬의 방위평균 σ 가 전체의 112% [^12] 다 — 코히런트 합이라 100 % 를 넘는다.
5. 그 광선 격자를 자세마다 다시 정의하면 로터 사이에 가짜 결합이 생긴다 — 가산성 잔차가 격자를 얼렸을 때 8.3e-16 [^13] (기계정밀도)이고 움직이는 격자에서 1.78 [^14] 다. 얼리면 대역밖 절대 전력이 λ/12 에서 13.1 dB [^15] 내려간다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| ① 조명면 찾기 | Sionna 의 Mitsuba/OptiX 광선엔진을 그대로 부른다 — 첫 충돌 탐색과 자기가림 판정이 그쪽 몫이다 |
| ② 면적분 | 그 면 위에서 부품별 재질 PO 를 적분한다 (`src/rcs_sbr.py` `rcs_sbr()`) — E = Σ \|Γᵢ(θᵢ)\| e^{j2k pᵢ·û} d², σ = 4π\|E\|²/λ² |
| 셸 투과 | 얇은 유전체 셸 뒤의 금속(배터리·PCB)을 코히런트 합산한다 (동 `penetrate=True`) |
| 가림의 크기 | 같은 자세에서 가림을 끄고 다시 적분해 방위평균 σ 의 차이를 기체마다 잰다 — 이산화 바닥과 나란히 싣는다 |
| 격자를 무엇에 매나 | 격자 중심·반경·칸수를 자세마다 다시 잡는 팔과 한 판으로 얼린 팔을 같은 씬·같은 자세열에 나란히 태운다 |

### 재현

```bash
PYTHONPATH=src python src/make_report02_target.py --derive-only
PYTHONPATH=src python src/build_part04_kernel.py
```

| | |
|---|---|
| 출력 | `outputs/report02_derived.json`, `outputs/prior_settled_sionna.json`, `outputs/report3_rt.json`, `outputs/sbr_grid_convergence.json`, `outputs/outofband_power.json`, `outputs/verify_frozen_grid.json`, `outputs/md_classify_verify.json` |
| 소요 | 약 2분 (GPU 0장 — 원장 조립이다) |
| 비고 | σ 격자 자체의 재생성은 `benchmark/rcs_anchor.py` 가 맡는다 |

---

## 두 낱말을 먼저 푼다

**PO** 는 물리광학(physical optics)이다 — 빛이 닿는 면에 흐르는 전류를 근사식으로 바로 적어 넣고 그 면을 훑어 더해 산란을 내는 방법이다. **SBR** 은 광선을 쏴서 튀기며 그 면이 어디인지 찾는 방법(shooting-and-bouncing rays)이다.

상용 고주파 RCS 솔버(FEKO/CST SBR+)의 순서 그대로다 — **① 광선으로 실제 조명면을 찾고 ② 그 위에서 PO 표면적분**(`src/rcs_sbr.py` `rcs_sbr()`). 레이다식이 표적 산란과 전파 경로를 두 양으로 쓰는 그대로, **σ 는 이 커널이 내고 경로와 환경은 그 엔진이 낸다**.

## 누가 무엇을 하나

| 단계 | 무엇을 | 누가 |
|---|---|---|
| 첫 충돌 탐색 · 가림 | 어느 면이 실제로 조명되는가 | 🟢 Sionna 의 Mitsuba/OptiX 광선엔진 |
| 재질 \|Γ(θ)\| | 수직입사 보정값 × 각도 모양(TE·TM 전력평균, `ANGLE_GAMMA=1` 기본) | 🟢 Sionna 재질표(`src/materials.py` `MATERIALS`) + 🔵 각도 모양 (`src/rcs_sbr.py` `ANGLE_GAMMA`) |
| PO 면적분 → σ | E = Σ \|Γᵢ(θᵢ)\| e^{j2k pᵢ·û} d², σ = 4π\|E\|²/λ² | 🔵 우리 (`src/rcs_sbr.py` `rcs_sbr()`) |
| 셸 투과 | 얇은 유전체 셸 뒤 금속(배터리·PCB)의 코히런트 합 | 🔵 우리 (동 `penetrate=True`) |

## 왜 우리가 얹어야 하나

Sionna 는 광선을 쏘고 튀긴다 — 기술보고서(v1.2, 59쪽)에 SBR 이 48 회 [^16] 나오고 우리도 그 엔진을 그대로 부른다. 같은 문서에서 `physical optics` 0 회 [^1] · `radar cross section` 0 회 [^17] · `surface current` 0 회 [^18] 이고, 거친 면은 정규화 산란패턴을 쓰는 경험 모델이다 — 그 셈이 어디서 끝나는지는 [부 1 «스톡 엔진이 하는 일과 안 하는 일»](../README.md#부-1-스톡-엔진이-하는-일과-안-하는-일) 가 인자 목록까지 해부했다.

ITU `metal` 의 산란계수 S = 0.0 [^19] 이라 스톡 산란 모델이 금속에서 내놓는 항은 0 이고, 우리 σ 는 면적분에서 창발한다. 금속 4그룹(모터·배터리·PCB·카메라)만 남긴 메쉬의 방위평균 σ 는 전체의 112% [^12] 다.

## PO 적분이 실제로 올라타는 면은 어디까지인가

![mesh_compare_material_shadow](../outputs/figures/mesh_compare_material_shadow.png)

**그림 1.** PO 적분이 실제로 올라타는 면은 어디까지인가?

조명원을 방위 280° [^2] · 고각 15° [^3] 에 두었다 — 방위 72 점 [^20] 스윕에서 7기체 평균 그늘비율의 중앙값에 가장 가까운 방위이고, 규칙이 고른다. 가림 판정은 생산 SBR 이 쓰는 그림자광선 그대로다(`rcs_sbr._exit_visible()`). 그림의 색은 재질이 아니라 조명 상태다.

| 기체 | 외피 그늘 | 가림 [dB] | 셸 투과 [dB] | 합 [dB] | 이산화 바닥 [dB] | 생산 σ [dBsm] |
|---|---|---|---|---|---|---|
| Mini 5 Pro ⭐ | 41 % | +5.98 | +3.66 | +2.32 | 0.014 | -22.0 |
| Mavic 4 Pro | 29 % | +3.75 | +2.55 | +1.20 | 0.021 | -18.2 |
| Matrice 4E ⭐ | 35 % | +6.63 | +3.72 | +2.90 | 0.021 | -18.9 |
| Phantom 4 | 36 % | +5.90 | +4.06 | +1.84 | 0.056 | -19.9 |
| X500 V2 | 47 % | +1.07 | +0.00 | +1.07 | 0.037 | -16.8 |
| Typhoon H (H480) | 39 % | +2.91 | +1.51 | +1.40 | 0.014 | -15.8 |
| S1000+ | 42 % | +0.11 | -0.19 | +0.30 | 0.071 | -12.3 |

출처 [^21]

## 가림을 끄면 얼마나 부푸나

가림을 끄면 방위평균 σ 가 6.63 dB [^6] (Matrice 4E ⭐ [^7], 닫힌 동체)까지 부풀고, 열린 프레임인 S1000+ [^8] 에서는 0.11 dB [^9] 다. 기체 7 종 [^10] 전부에서 이 값이 이산화 바닥(최대 0.071 dB [^11]) 위에 있다.

⚠ 이 표와 그림은 2026-08-04 [^22] 형상 정정 **전** 메쉬 기준이고, 2026-08-07 10:58:22 [^23] Γ(θ) 각도 모양(기본 켬) **이전** 커널의 산출이다 — 가림 최대치를 내는 Matrice 4E 와 X500 V2 가 그 정정을 받은 기체이고, 닫힌 동체의 가림은 셸 형상에 직접 걸린다. 생산 σ 열도 두 축 같은 이유로 재계산 대상이다.

## 격자를 자세마다 다시 정의하면 무엇이 생기나

광선 격자는 표적 앞에 세우는 평면 자다 — 중심 ctr, 반경 Rout, 한 변의 칸수 n 셋이 그것을 정한다. 생산 경로는 그 셋을 **자세마다 bbox 에서 다시 잡는다**. 관절이 도는 로터에서는 bbox 가 자세마다 숨쉬므로 자도 같이 흔들린다.

| 흔들리는 것 | 무엇이 흔들리나 (λ/12 · matrice4e · 4096 자세) | 무엇이 실리나 |
|---|---|---|
| 위상 원점 | ctr 이 시선방향으로 39.9 mm [^24] p-p 돌아다닌다 = 5.85 rad [^25] p-p | 진폭은 4e-16 [^26] 안에서 불변인 채 **위상만** 흔들린다 — 정지한 동체를 시선방향으로 숨쉬게 만드는 것과 같다 |
| 표본 격자 | n = ceil(2Rout/d) 가 정수라 100 [^27]~131 [^28] 사이를 오가고 4095 스텝 중 1636 번 [^29] (40 %) 튄다 | 서브셀 오프셋 표준편차 0.2912 [^30] 는 균등분포 1/√12 = 0.2887 과 넷째 자리까지 같다 — 자세마다 굴리는 **백색 주사위**다 |
| 히트 집합 | 조명된 광선이 평균 610.5 개 [^31], 자세간 상대 표준편차 0.0378 [^32] | 자세별 히트 수 계열 [^33] 의 최대−최소가 102 개(17 %) 다 — 어느 면이 세어지는가가 자세마다 갈린다 |

## 결정적 검사 — 가산성

서로 가리지 않는 로터의 PO 면적분은 E(φ₁..φ₄) = E₀ + Σ ΔE_j(φ_j) 로 정확히 쪼개진다. 로터 넷을 따로 돌린 합과 넷을 함께 돌린 장의 차이를 잔차로 쓴다 — 이 잣대에는 창도 평활도 분모도 안 들어간다.

| 격자 | 가산성 잔차 (중앙값) | 읽는 법 |
|---|---|---|
| 얼린 판 한 장 | 8.3e-16 [^13] ~ 1.2e-15 [^34] | 기계정밀도 — 정리가 그대로 성립한다 |
| 자세마다 다시 정의 | 0.089 [^35] ~ 1.78 [^14] | O(1) — 물리적으로 결합할 수 없는 로터 사이에 결합이 생긴다 |

그 가짜 결합이 변조로 실린다 — 기체별로 +3.7 [^36] ~ +23.3 dB [^37] 다. 교차 증거로, 광선을 안 쓰는 독립 엔진(순수 PO)과의 대역 안 스펙트럼 일치가 0.440 [^38] 에서 0.953 [^39] 으로 오른다.

## 얼리면 무엇이 오고 무엇을 잃나

판 하나를 잡아 4096 자세에 그대로 쓰면 슬로타임 스펙트럼의 대역밖 절대 전력이 λ/12 에서 13.1 [^15] · λ/32 에서 20.1 dB [^40] 내려간다. ⭐ 그리고 **얼린 팔만 예측대로 d² 로 수렴한다** — 기울기 -2.19 [^41] (R² 0.998 [^42]) 대 생산 팔 -0.56 [^43] (R² 0.946 [^44]) 다. 생산 격자는 λ/12 → λ/32 로 촘촘히 해도 2.3 dB [^45] 만 내려간다 — 바닥의 지배 원인이 광선 밀도가 아니라는 뜻이다.

| 대가 | 크기 | 무엇을 뜻하나 |
|---|---|---|
| 광선 수 | 얼린 판이 자세 평균 대비 1.108 배 [^46] | 전 자세를 덮는 판이라 평균보다 크다 — 비용 +10.8 % |
| 디더 평균 | 얼린 장과 생산 장의 레벨 차가 3.35 dB [^47] p-p | 자세별 무작위 오프셋은 사실상 몬테카를로 평균이다. 얼리면 오프셋 한 판에 절대 레벨이 걸린다 — 절대 σ 는 정적 경로에서 가져오고 얼린 복소장은 **모양**에만 쓴다 |
| 판을 미리 잡는 일 | 덮개 여유 최소 120.5 mm [^48] | 자세열을 먼저 훑어야 판이 나온다 — 스트리밍으로는 못 잡는다 |

배선은 커널에 들어가 있고 기본값은 `grid_ref=None` 이다 — 그 값이면 배선 전 커널과 36 [^49]/36 [^50] 비트 동일이라 (최대 상대오차 0.0 [^51]) 기존 원장이 그대로 선다. 하류(`src/microdoppler.py` · 리포트 8 계열)는 아직 `grid_ref` 를 안 넘긴다 — 이 절의 이득은 커널의 성질이지 지금 원장의 숫자가 아니다.

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 하류 마이크로도플러 경로에 `grid_ref` 를 넘기고 원장을 다시 낸다 | 얼린 격자의 이득이 리포트 8 계열의 숫자로 들어온다 | `src/microdoppler.py` → [편 35 «시간표본마다 자세를 새로 놓고 다시 쏘아 슬로…»](35_md-slowtime.ipynb) |
| 정정된 메쉬로 가림 표와 생산 σ 를 같은 설정에서 다시 낸다 | 형상 정정이 가림과 σ 를 어느 방향으로 얼마나 옮기는지가 기체별로 확정된다 | [^52] |
| 같은 메쉬를 스톡 경로 솔버에 그대로 넣고 무엇이 나오는지 잰다 | 우리 커널이 스톡 위에 얹은 항이 무엇인지가 나란히 확정된다 | [편 19 «스톡 솔버와 맞대면 «면이 많아서 에코가 커진…»](19_kernel-vs-stock.ipynb) |
| 수신 방향 그림자 광선을 켜고 바이스태틱으로 넓힌다 | 출사 쪽 가림이 상반성 위반을 얼마나 줄이는지가 확정된다 | [편 20 «수신 방향 그림자 광선을 켜면 상반성 위반이…»](20_bistatic-exit.ipynb) |
| PO 면적분을 디바이스 커널로 옮긴다 | 전격자 재생성 비용이 확정된다 — 지금은 호스트가 대부분을 쓴다 | [편 19 «스톡 솔버와 맞대면 «면이 많아서 에코가 커진…»](19_kernel-vs-stock.ipynb) |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 52개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/prior_settled_sionna.json` | `word_counts_rerun_this_session.sionna_rt_technical_report_v2_59p.physical optics` | 0 |
| [^2] | `outputs/report02_derived.json` | `occlusion.az_deg` | 280 |
| [^3] | `outputs/report02_derived.json` | `occlusion.el_deg` | 15 |
| [^4] | `outputs/report02_derived.json` | `occlusion.shadow_min_pct` | 29.06 |
| [^5] | `outputs/report02_derived.json` | `occlusion.shadow_max_pct` | 46.54 |
| [^6] | `outputs/report02_derived.json` | `occlusion.max_db` | 6.626 |
| [^7] | `outputs/report02_derived.json` | `occlusion.max_drone` | Matrice 4E ⭐ |
| [^8] | `outputs/report02_derived.json` | `occlusion.min_drone` | S1000+ |
| [^9] | `outputs/report02_derived.json` | `occlusion.min_db` | 0.1111 |
| [^10] | `outputs/report02_derived.json` | `occlusion.n_above_floor` | 7 |
| [^11] | `outputs/report02_derived.json` | `occlusion.floor_max_db` | 0.07061 |
| [^12] | `outputs/report3_rt.json` | `C_metal.metal_share_pct` | 112 |
| [^13] | `outputs/md_classify_verify.json` | `grid_pinning.matrice4e.pinned.additivity_residual_median` | 8.282e-16 |
| [^14] | `outputs/md_classify_verify.json` | `grid_pinning.matrice4e.moving.additivity_residual_median` | 1.781 |
| [^15] | `outputs/outofband_power.json` | `freeze_verdict.gains_db.12` | 13.09 |
| [^16] | `outputs/prior_settled_sionna.json` | `word_counts_rerun_this_session.sionna_rt_technical_report_v2_59p.SBR or shooting-and-bouncing` | 48 |
| [^17] | `outputs/prior_settled_sionna.json` | `word_counts_rerun_this_session.sionna_rt_technical_report_v2_59p.radar cross section` | 0 |
| [^18] | `outputs/prior_settled_sionna.json` | `word_counts_rerun_this_session.sionna_rt_technical_report_v2_59p.surface current` | 0 |
| [^19] | `outputs/report3_rt.json` | `C_metal.itu_metal_S` | 0 |
| [^20] | `outputs/report02_derived.json` | `occlusion.n_az_sweep` | 72 |
| [^21] | `outputs/report02_derived.json` | `occlusion.rows` | (7행 표) |
| [^22] | `outputs/meshfix_applied.json` | `_meta.date` | 2026-08-04 |
| [^23] | `outputs/angle_gamma_impact.json` | `_meta.generated` | 2026-08-07 10:58:22 |
| [^24] | `outputs/sbr_grid_convergence.json` | `grid_wander.ctr_u_ptp_mm` | 39.89 |
| [^25] | `outputs/sbr_grid_freeze_review.json` | `R4_phase_only_arm.ctr_dot_u_ptp_rad` | 5.852 |
| [^26] | `outputs/sbr_grid_freeze_review.json` | `R4_phase_only_arm.rows[1].max_abs_change` | 3.906e-16 |
| [^27] | `outputs/sbr_grid_convergence.json` | `grid_wander.per_div.12.n_min` | 100 |
| [^28] | `outputs/sbr_grid_convergence.json` | `grid_wander.per_div.12.n_max` | 131 |
| [^29] | `outputs/sbr_grid_convergence.json` | `grid_wander.per_div.12.n_changes` | 1636 |
| [^30] | `outputs/sbr_grid_convergence.json` | `grid_wander.per_div.12.subcell_off_e1_std_frac` | 0.2912 |
| [^31] | `outputs/adv_grid_freeze_audit.json` | `audit_2_signal_loss.rows[1].n_lit_prod_mean` | 610.5 |
| [^32] | `outputs/adv_grid_freeze_audit.json` | `audit_2_signal_loss.rows[1].n_lit_prod_relstd` | 0.03777 |
| [^33] | `outputs/sbr_grid_convergence.npz` | `n_lit_div12` | (4096행 표) |
| [^34] | `outputs/md_classify_verify.json` | `grid_pinning.mini5pro.pinned.additivity_residual_median` | 1.167e-15 |
| [^35] | `outputs/md_classify_verify.json` | `grid_pinning.s1000plus.moving.additivity_residual_median` | 0.08905 |
| [^36] | `outputs/md_classify_verify.json` | `grid_pinning.matrice4e.spurious_modulation_db` | 3.651 |
| [^37] | `outputs/md_classify_verify.json` | `grid_pinning.phantom4.spurious_modulation_db` | 23.31 |
| [^38] | `outputs/sbr_grid_convergence.json` | `in_band_fidelity.rows[1].cos_prod_vs_po` | 0.4402 |
| [^39] | `outputs/sbr_grid_convergence.json` | `in_band_fidelity.rows[1].cos_froz_vs_po` | 0.9535 |
| [^40] | `outputs/outofband_power.json` | `freeze_verdict.gains_db.32` | 20.13 |
| [^41] | `outputs/outofband_power.json` | `convergence.froz.slope_ge12` | -2.191 |
| [^42] | `outputs/outofband_power.json` | `convergence.froz.r2_ge12` | 0.9981 |
| [^43] | `outputs/outofband_power.json` | `convergence.prod.slope_ge12` | -0.5604 |
| [^44] | `outputs/outofband_power.json` | `convergence.prod.r2_ge12` | 0.9463 |
| [^45] | `outputs/outofband_power.json` | `convergence.prod.drop_db_div12_to_div32` | 2.297 |
| [^46] | `outputs/verify_frozen_grid.json` | `gate2_frozen_grid_invariant.extra_ray_cost` | 1.108 |
| [^47] | `outputs/verify_frozen_grid.json` | `field_level.froz_vs_prod_level_db_ptp` | 3.351 |
| [^48] | `outputs/verify_frozen_grid.json` | `gate3_coverage.margin_min_mm` | 120.5 |
| [^49] | `outputs/verify_frozen_grid.json` | `gate1_bit_identity.n_bit_identical` | 36 |
| [^50] | `outputs/verify_frozen_grid.json` | `gate1_bit_identity.n_cases` | 36 |
| [^51] | `outputs/verify_frozen_grid.json` | `gate1_bit_identity.max_rel_err` | 0 |
| [^52] | `outputs/meshfix_attack.json` | `recommended_gate_before_any_sigma_claim` | (6행 표) |